---

# Submission Kaggle: SVM dengan Kernel yang Dipelajari

*Satu SVM from scratch (slack kuadrat) dengan kernel Hamming berbobot. Bobot kernel dipelajari dari data latih.*

---

## Kelompok G04: TreeIsAllYouNeed

Alur notebook ini:

1. Fitur dibuat dari data latih saja, lalu diterapkan ke data uji.
2. Tiap fitur diubah menjadi kode termometer, yaitu bit "x > ambang".
3. Bobot tiap bit dipelajari dengan gradien galat holdout (`src/kaggle/kernel_learning.py`).
4. SVM dilatih pada seluruh data latih (`src/algoritma/svm/l2_svm.py`), disimpan, dimuat ulang, lalu dipakai untuk
   membuat submission.

Hanya NumPy, SciPy (faktorisasi Cholesky), dan pandas yang dipakai. Tidak ada ensemble dan tidak ada data luar.

---

# Inisialisasi <a name="1"></a>

---

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "kaggle" / "dataset" / "train.csv").exists())
for folder in ("algoritma/svm", "kaggle"):
    sys.path.insert(0, str(REPO / "src" / folder))

from booking_features import build_features, clean, feature_weights, load_data, standardize
from kernel_learning import SCALE, learn_kernel, thermometer
from l2_svm import L2SVM

DATA = REPO / "src" / "kaggle" / "dataset"
OUTPUT = REPO / "output" / "svm"
OUTPUT.mkdir(parents=True, exist_ok=True)
WEIGHTS = OUTPUT / "kernel_weights.npz"
THRESHOLD = -0.10

---

# Fitur dan Kode Termometer <a name="2"></a>

---

Fitur sama dengan `svm_notebook.ipynb` (46 kolom). Bobot AGOP dari kernel Laplacian dipakai sebagai titik awal.
Ambang termometer adalah titik tengah antarnilai unik, atau titik kuantil bila nilai uniknya lebih dari 128.

In [2]:
train_raw, test = load_data(DATA)
train = clean(train_raw)
y = train["is_canceled"].to_numpy()
X, X_test = build_features(train, y, test, shared_neighbours=True)
Z, _, names = standardize(X, X_test)
agop = feature_weights(Z, y, SCALE)
codes, codes_test, start, bit_names = thermometer(X, X_test, agop)
print("fitur:", X.shape[1], " bit termometer:", codes.shape[1], " baris latih:", len(codes))

fitur: 46  bit termometer: 792  baris latih: 27309


---

# Belajar Kernel <a name="3"></a>

---

Kernel berbentuk K(x, z) = jumlah a_m exp(-jumlah w_mj |b_j(x) - b_j(z)|) dengan dua komponen. Tiap langkah
mengambil 4.000 baris untuk melatih dan 3.000 baris lain untuk menghitung galat, lalu bobot digeser dengan Adam.

Hasil belajar disimpan di `output/svm/kernel_weights.npz`. Pembulatan floating point bisa sedikit berbeda antar
mesin, sehingga model akhir memakai bobot tersimpan agar submission selalu sama.

In [3]:
started = time.time()
weights, mixture, reg = learn_kernel(codes, y, start)
print(f"{time.time() - started:.0f} detik")
if not WEIGHTS.exists():
    np.savez(WEIGHTS, weights=np.array(weights), mixture=mixture, reg=reg)
saved = np.load(WEIGHTS)
gap = max(np.abs(np.log(a) - np.log(b)).max() for a, b in zip(weights, saved["weights"]))
print("selisih log bobot terbesar terhadap bobot tersimpan:", round(float(gap), 4))
weights, mixture, reg = list(saved["weights"]), saved["mixture"], float(saved["reg"])
print("campuran:", mixture.round(3), " regularisasi:", round(reg, 3))

langkah 20: galat kuadrat holdout 0.3701, reg 0.828, campuran [0.689 0.311]


langkah 40: galat kuadrat holdout 0.3564, reg 0.971, campuran [0.673 0.327]


langkah 60: galat kuadrat holdout 0.3463, reg 1.021, campuran [0.692 0.308]


langkah 80: galat kuadrat holdout 0.3400, reg 0.980, campuran [0.721 0.279]


langkah 100: galat kuadrat holdout 0.3340, reg 0.927, campuran [0.759 0.241]


57 detik
selisih log bobot terbesar terhadap bobot tersimpan: 0.0
campuran: [0.73 0.27]  regularisasi: 0.969


In [4]:
source = [name.split(" > ")[0] for name in bit_names]
pd.DataFrame({f"komponen {m + 1}": pd.Series(w).groupby(source).sum() for m, w in enumerate(weights)}
             ).sort_values("komponen 1", ascending=False).head(12).round(2)

,komponen 1,komponen 2
special_requests,2.60,6.22
sib_te,1.68,7.27
nightly_rate_idr,1.55,3.72
arrival_month,1.26,2.92
days_before_arrival,1.18,4.73
book_ord,0.68,2.79
arr_ord,0.53,1.89
weekday_nights,0.46,0.44
needs_parking,0.42,1.45
num_children,0.32,1.44


---

# Pelatihan dan Submission <a name="4"></a>

---

SVM memakai slack kuadrat dan dilatih dengan metode Newton pada himpunan support vector. Ambang keputusan
-0,10 dipilih dari prediksi out-of-fold data latih.

In [5]:
started = time.time()
svm = L2SVM(weights, mixture, reg=reg).fit(codes, y)
print(f"iterasi Newton: {svm.n_iter_}, support vector: {len(svm.support_)}, {time.time() - started:.0f} detik")

svm.save(OUTPUT / "svm_kernel_model.pkl")
svm = L2SVM.load(OUTPUT / "svm_kernel_model.pkl")

scores = svm.decision_function(codes_test)
submission = pd.DataFrame({"id": test["id"], "is_canceled": (scores > THRESHOLD).astype(int)})
sample = pd.read_csv(DATA / "sample_submission.csv", dtype={"id": str})
assert list(submission.columns) == list(sample.columns)
assert submission["id"].tolist() == sample["id"].tolist()
submission.to_csv(OUTPUT / "submission.csv", index=False)
print("baris:", len(submission), " rasio batal:", round(submission["is_canceled"].mean(), 4),
      " jarak skor terdekat ke ambang:", float(np.abs(scores - THRESHOLD).min()))

iterasi Newton: 7, support vector: 16243, 102 detik


baris: 6844  rasio batal: 0.3087  jarak skor terdekat ke ambang: 0.0002321664467598572


---

# Submission Kedua <a name="5"></a>

---

Penilaian akhir Kaggle memakai submission terbaik dari paling banyak tiga pilihan. Submission kedua memakai kode
dan langkah yang sama, dengan 255 kuantil, tiga komponen kernel, dan seed lain. Model ini berdiri sendiri dan tidak
digabung dengan model pertama.

In [6]:
codes_b, codes_test_b, start_b, _ = thermometer(X, X_test, agop, bins=255)
started = time.time()
weights_b, mixture_b, reg_b = learn_kernel(codes_b, y, start_b, seed=1, scales=(0.7, 1.5, 3.0))
print(f"bit termometer: {codes_b.shape[1]}, {time.time() - started:.0f} detik")
path_b = OUTPUT / "kernel_weights_b.npz"
if not path_b.exists():
    np.savez(path_b, weights=np.array(weights_b), mixture=mixture_b, reg=reg_b)
saved = np.load(path_b)
gap = max(np.abs(np.log(a) - np.log(b)).max() for a, b in zip(weights_b, saved["weights"]))
print("selisih log bobot terbesar terhadap bobot tersimpan:", round(float(gap), 4))

langkah 20: galat kuadrat holdout 0.3800, reg 0.815, campuran [0.536 0.211 0.252]


langkah 40: galat kuadrat holdout 0.3612, reg 0.913, campuran [0.569 0.159 0.272]


langkah 60: galat kuadrat holdout 0.3468, reg 0.922, campuran [0.612 0.128 0.26 ]


langkah 80: galat kuadrat holdout 0.3416, reg 0.905, campuran [0.646 0.133 0.221]


langkah 100: galat kuadrat holdout 0.3374, reg 0.875, campuran [0.618 0.155 0.227]


bit termometer: 1295, 95 detik
selisih log bobot terbesar terhadap bobot tersimpan: 0.0


In [7]:
started = time.time()
svm_b = L2SVM(list(saved["weights"]), saved["mixture"], reg=float(saved["reg"])).fit(codes_b, y)
print(f"iterasi Newton: {svm_b.n_iter_}, support vector: {len(svm_b.support_)}, {time.time() - started:.0f} detik")
svm_b.save(OUTPUT / "svm_kernel_model_b.pkl")
svm_b = L2SVM.load(OUTPUT / "svm_kernel_model_b.pkl")

scores_b = svm_b.decision_function(codes_test_b)
submission_b = pd.DataFrame({"id": test["id"], "is_canceled": (scores_b > THRESHOLD).astype(int)})
submission_b.to_csv(OUTPUT / "submission_b.csv", index=False)
print("baris:", len(submission_b), " rasio batal:", round(submission_b["is_canceled"].mean(), 4),
      " label beda dari submission pertama:", int((submission_b["is_canceled"] != submission["is_canceled"]).sum()),
      " jarak skor terdekat ke ambang:", float(np.abs(scores_b - THRESHOLD).min()))

iterasi Newton: 7, support vector: 16222, 112 detik


baris: 6844  rasio batal: 0.3083  label beda dari submission pertama: 59  jarak skor terdekat ke ambang: 0.0002019256163340588


---

# Referensi <a name="6"></a>

---

1. O. Chapelle, "Training a Support Vector Machine in the Primal", Neural Computation 19(5), 2007.
2. O. Chapelle, V. Vapnik, O. Bousquet, dan S. Mukherjee, "Choosing Multiple Parameters for Support Vector
   Machines", Machine Learning 46, 2002.
3. H. Owhadi dan G. R. Yoo, "Kernel Flows", Journal of Computational Physics 389, 2019.
4. G. Lanckriet dkk., "Learning the Kernel Matrix with Semidefinite Programming", JMLR 5, 2004.
5. D. Kingma dan J. Ba, "Adam: A Method for Stochastic Optimization", ICLR 2015.
6. M. Hestenes dan E. Stiefel, "Methods of Conjugate Gradients for Solving Linear Systems", J. Res. NBS 49(6), 1952.
7. Y. Gorishniy, I. Rubachev, dan A. Babenko, "On Embeddings for Numerical Features in Tabular Deep Learning",
   NeurIPS 2022.
8. A. Radhakrishnan dkk., "Mechanism for feature learning in neural networks and backpropagation-free machine
   learning models", Science, 2024 (bobot awal).